# Chapter 9 Lab: Equiangular Frames and Grassmannian Packings

This notebook accompanies **Chapter 9**. It builds coherence, the Welch
bound, and the ETF checker (with the corrected max/min comparison, not
the ordering-dependent `off[0]` version), plus the Chapter 8 gradient
descent machinery, then works through all six lab exercises.

Run the setup cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

def frame_potential(V):
    S = V @ V.T
    return np.trace(S @ S)

def fp_gradient(V):
    S = V @ V.T
    return 4 * S @ V

def normalize_columns(V):
    norms = np.linalg.norm(V, axis=0)
    return V / norms

def fit_tight_frame(n, m, num_steps=500, lr=0.05, seed=0):
    rng = np.random.default_rng(seed)
    V = rng.standard_normal((n, m))
    V = normalize_columns(V)
    history = [frame_potential(V)]
    for step in range(num_steps):
        grad = fp_gradient(V)
        V = V - lr * grad
        V = normalize_columns(V)
        history.append(frame_potential(V))
    return V, history

def gram_matrix(F):
    '''Gram matrix G = F^* F (shape m x m).'''
    return F.conj().T @ F

def coherence(F):
    G = gram_matrix(F)
    off_diag = np.abs(G - np.diag(np.diag(G)))
    return off_diag.max()

def welch_bound(n, m):
    return np.sqrt((m - n) / (n * (m - 1)))

def is_etf(F, tol=1e-6):
    '''Returns (is_tight, is_equiangular, achieves_welch).'''
    n, m = F.shape
    S = F @ F.T.conj()
    eigs = np.linalg.eigvalsh(S.real)
    tight = bool(np.isclose(eigs.min(), eigs.max(), atol=tol))
    G = gram_matrix(F)
    off = np.array([np.abs(G[i,j]) for i in range(m) for j in range(m) if i != j])
    equi = bool(np.isclose(off.max(), off.min(), atol=tol))
    mu = coherence(F)
    achieves = bool(np.isclose(mu, welch_bound(n, m), atol=tol))
    return tight, equi, achieves

def regular_polygon(m):
    angles = 2 * np.pi * np.arange(m) / m
    return np.vstack([np.cos(angles), np.sin(angles)])

## Lab Exercise 9.1: Regular Polygons and Equiangularity

Run `is\_etf` on all regular $m$-gons in $\mathbb{R}^2$ for $m=3,\ldots,12$. Confirm only $m=3$ is an ETF; explain why even-$m$ fails (hint: $\mu=1$).

In [ ]:
print(f"{'m':>3} {'mu':>8} {'Welch':>8} {'tight':>7} {'equi':>6} {'ETF':>5}")
for m in range(3, 13):
    F = regular_polygon(m)
    mu = None       # TODO: coherence(F)
    wb = None       # TODO: welch_bound(2, m)
    tight, equi, achieves = None, None, None  # TODO: is_etf(F)
    print(f"{m:>3} {mu:>8.4f} {wb:>8.4f} {str(tight):>7} {str(equi):>6} {str(tight and equi):>5}")

*Your geometric explanation: why does mu=1 occur for even m, and why does this break equiangularity so dramatically?* (replace this text)

## Lab Exercise 9.2: Welch Bound Survey

For $(n,m)\in\{(2,3),(2,5),(3,4),(3,6),(3,7),(4,6),(4,7)\}$: compute Welch bound, fit a tight frame, report coherence excess. Identify pairs that admit an ETF.

In [ ]:
pairs = [(2,3), (2,5), (3,4), (3,6), (3,7), (4,6), (4,7)]

print(f"{'(n,m)':>8} {'Welch':>8} {'mu(fitted)':>11} {'excess':>8}")
for n, m in pairs:
    wb = None  # TODO: welch_bound(n, m)
    V, _ = None, None  # TODO: fit_tight_frame(n, m, num_steps=1500, lr=0.02, seed=0)
    mu_fitted = None  # TODO: coherence(V)
    excess = mu_fitted / wb
    print(f"({n},{m})".rjust(8) + f" {wb:>8.4f} {mu_fitted:>11.4f} {excess:>8.4f}")

*Your identification: which (n,m) pairs appear to admit an ETF (excess close to 1)?* (replace this text)

## Lab Exercise 9.3: The Gram Matrix Condition

Verify $G^2=(3/2)G$ for the Mercedes-Benz frame. Construct a different Gram matrix $G'$ with the same diagonal/off-diagonal magnitudes but permuted signs. Is it PSD? Does it correspond to a valid frame?

In [ ]:
f1 = np.array([1.0, 0.0])
f2 = np.array([-0.5, np.sqrt(3)/2])
f3 = np.array([-0.5, -np.sqrt(3)/2])
F_mb = np.column_stack([f1, f2, f3])

G_mb = gram_matrix(F_mb)
check = None  # TODO: np.allclose(G_mb @ G_mb, (3/2) * G_mb)
print(f"G_mb^2 == (3/2) G_mb: {check}")

G_prime = np.array([
    [1.0,  0.5,  0.5],
    [0.5,  1.0,  0.5],
    [0.5,  0.5,  1.0],
])

check_prime = None  # TODO: verify G_prime^2 == (3/2) * G_prime
print(f"G_prime^2 == (3/2) G_prime: {check_prime}")

eigs_prime = None  # TODO: np.linalg.eigvalsh(G_prime)
print(f"Eigenvalues of G_prime: {eigs_prime}")
is_psd = None  # TODO: np.all(eigs_prime >= -1e-10)
print(f"Is G_prime PSD? {is_psd}")

if is_psd:
    eigvals, eigvecs = np.linalg.eigh(G_prime)
    eigvals = np.clip(eigvals, 0, None)
    F_prime = np.diag(np.sqrt(eigvals)) @ eigvecs.T
    print(f"\nExtracted frame vectors (columns), shape {F_prime.shape}:\n{F_prime}")
    print(f"Check: F_prime^T @ F_prime == G_prime? {np.allclose(F_prime.T @ F_prime, G_prime)}")

*Your conclusion: does G' correspond to a valid frame? What does this say about non-congruent ETF configurations sharing the same equiangularity value?* (replace this text)

## Lab Exercise 9.4: Coherence-Minimizing Descent

Minimize the soft-max coherence approximation with $\alpha=8$ for $(n,m)=(2,3)$ and $(3,4)$. Compare to the Welch bound and to frame-potential minimization.

In [ ]:
def soft_coherence(V, alpha=8):
    G = V.T @ V
    m = V.shape[1]
    off_diag_sq = (G - np.diag(np.diag(G)))**2
    return np.sum(off_diag_sq**alpha)**(1/(2*alpha))

def soft_coherence_gradient(V, alpha=8, eps=1e-8):
    grad = np.zeros_like(V)
    base = soft_coherence(V, alpha)
    for i in range(V.shape[0]):
        for j in range(V.shape[1]):
            V_pert = V.copy()
            V_pert[i, j] += eps
            grad[i, j] = (soft_coherence(V_pert, alpha) - base) / eps
    return grad

def fit_min_coherence(n, m, num_steps=300, lr=0.05, alpha=8, seed=0):
    rng = np.random.default_rng(seed)
    V = rng.standard_normal((n, m))
    V = normalize_columns(V)
    for step in range(num_steps):
        grad = None  # TODO: soft_coherence_gradient(V, alpha=alpha)
        V = V - lr * grad
        V = normalize_columns(V)
    return V

for n, m in [(2, 3), (3, 4)]:
    V_coh = fit_min_coherence(n, m, num_steps=300, lr=0.1, alpha=8, seed=0)
    mu_coh = coherence(V_coh)
    wb = welch_bound(n, m)
    print(f"(n,m)=({n},{m}): coherence-minimized mu={mu_coh:.4f}, Welch={wb:.4f}, "
          f"excess={mu_coh/wb:.4f}")

    V_fp, _ = fit_tight_frame(n, m, num_steps=1500, lr=0.02, seed=0)
    mu_fp = coherence(V_fp)
    print(f"                  FP-minimized mu={mu_fp:.4f}, excess={mu_fp/wb:.4f}\n")

*Your comparison: does directly minimizing coherence get closer to the Welch bound than minimizing frame potential?* (replace this text)

## Lab Exercise 9.5 (Optional, Challenge): ETFs in R^3

The six vectors $\{\pm e_1,\pm e_2,\pm e_3\}$ are tight ($A=2$) but check equiangularity. Research the smallest real ETF in $\mathbb{R}^3$ with $m>3$.

In [ ]:
e1, e2, e3 = np.eye(3)[:,0], np.eye(3)[:,1], np.eye(3)[:,2]
F_octahedron = np.column_stack([e1, -e1, e2, -e2, e3, -e3])

tight_oct, equi_oct, _ = None, None, None  # TODO: is_etf(F_octahedron)
print(f"{{+-e1,+-e2,+-e3}}: tight={tight_oct}, equiangular={equi_oct}")

G_oct = gram_matrix(F_octahedron)
off_oct = np.abs(G_oct - np.diag(np.diag(G_oct)))
print(f"Distinct off-diagonal magnitudes: {sorted(set(off_oct.round(4).flatten()) - {0.0})}")


*Your research: what is the smallest real ETF in R^3 with m>3? Report your source and verify it numerically above.* (replace this text)

## Lab Exercise 9.6 (Optional): Harmonic Frames in C^n

(a) $n=3$ DFT orthonormal basis. (b) $N=6$, subset $\{0,1,2\}$: tight? equiangular? (c) challenge: $N=7$, subset $\{0,1,3\}$ (a $(7,3,1)$ difference set).

In [ ]:
n = 3
omega3 = np.exp(2j * np.pi / n)
F_dft3 = np.array([[omega3**(j*k) / np.sqrt(n) for k in range(n)] for j in range(n)])

gram3 = np.zeros((n,n), dtype=complex)
for j in range(n):
    for k in range(n):
        gram3[j,k] = None  # TODO: np.vdot(F_dft3[:,j], F_dft3[:,k])
print("Part (a): Gram matrix (should be I):\n", gram3.round(6))

N = 6
omega6 = np.exp(2j * np.pi / N)
F_dft6_full = np.array([[omega6**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
F_dft6 = F_dft6_full / np.linalg.norm(F_dft6_full, axis=0)

S_subset = [0, 1, 2]
F_sub = F_dft6[:, S_subset]

tight_sub, equi_sub, _ = None, None, None  # TODO: is_etf(F_sub)
mu_sub = None       # TODO: coherence(F_sub)
wb_sub = None       # TODO: welch_bound(6, 3)
print(f"\nPart (b): tight={tight_sub}, equiangular={equi_sub}, mu={mu_sub:.4f}, Welch={wb_sub:.4f}")

In [ ]:
N7 = 7
omega7 = np.exp(2j * np.pi / N7)
F_dft7_full = np.array([[omega7**(j*k) / np.sqrt(N7) for k in range(N7)] for j in range(N7)])
F_dft7 = F_dft7_full / np.linalg.norm(F_dft7_full, axis=0)

S7 = [0, 1, 3]
F_sub7 = F_dft7[:, S7]

tight7, equi7, achieves7 = None, None, None  # TODO: is_etf(F_sub7)
print(f"S={{0,1,3}} in N=7 DFT: tight={tight7}, equiangular={equi7}, achieves_welch={achieves7}")